In [6]:
import os
import google.genai as genai
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv("GOOGLE_API_KEY")
gemini_model = "gemini-2.5-flash-lite"
gemini_client = genai.Client(api_key=gemini_key)

In [7]:
from scraper import fetch_website_contents

In [8]:
# This is the task where we are bulding the brochure ->

system_message = """
You are an assisstant that analyzes the contents of a company website landing page
and creates a short brochure about the company for perspective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [10]:
# Using stream instead of gradio ->
# Gemini ->
from openai import OpenAI

gemini_client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = gemini_client.chat.completions.create(
        model=gemini_model,
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result      
                
# Ollama ->
import ollama
def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = ollama.chat(
        model="llama3.2:3b",
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        result += chunk["message"]["content"]
        yield result

In [11]:
def stream_brochure(company_name,url,model):
        yield ""
        prompt = f"Generating a company brochure for {company_name}. Here is the content of the Landing Page: \n"
        prompt += fetch_website_contents(url)
        if model == "Gemini":
                result = stream_gemini(prompt)
        elif model == "Ollama":
                result = stream_ollama(prompt)
        else:
                raise ValueError("Unknown model")
        yield from result

In [12]:
import gradio as gr
name_input = gr.Textbox(label="Company Name: ")
url_input = gr.Textbox(label="Landing page url including http:// or https://")
model_selector = gr.Dropdown(["Gemini","Ollama"],label="Select model",value="Gemini")
message_output = gr.Markdown(label="Response: ")

In [13]:
view = gr.Interface(fn=stream_brochure,title="Brochure Generator",inputs=[name_input,url_input,model_selector],outputs=[message_output],examples=[
        ["Hugging Face","https://huggingface.co","Gemini"],
        ["Aditya Soni","https://adityasoni.dev","Ollama"]
],flagging_mode="never")
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
